
# Short Demo Guide

This notebook gives a compact walkthrough of two experiment scripts:

- `experiments/sda-bench.py` compares **Simple Dual Averaging (SDA)** with the **projected subgradient method** on built-in objective functions.
- `experiments/sda-logreg.py` runs **SDA**, **projected subgradient**, and an **sklearn logistic-regression baseline** on CSV datasets.

The guide uses both Python `help(...)` and real CLI `--help`, runs small examples, shows where outputs are written, and renders a few generated plots inline.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()


def load_module(path: str, module_name: str):
    module_path = ROOT / path
    parent = str(module_path.parent)
    src_root = str(ROOT / 'src')
    if parent not in sys.path:
        sys.path.insert(0, parent)
    if src_root not in sys.path:
        sys.path.insert(0, src_root)
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    module = importlib.util.module_from_spec(spec)
    assert spec is not None and spec.loader is not None
    spec.loader.exec_module(module)
    return module


def show_png(path: str):
    png_path = ROOT / path
    print(png_path)
    display(Image(filename=str(png_path)))


def show_json_keys(path: str):
    json_path = ROOT / path
    payload = json.loads(json_path.read_text(encoding='utf-8'))
    print(json_path)
    if isinstance(payload, list):
        print(f"entries: {len(payload)}")
        if payload:
            print('methods:', [entry.get('method') for entry in payload])
            print('first entry keys:', list(payload[0].keys())[:20])
    else:
        print('keys:', list(payload.keys())[:20])
    return payload


In [ ]:
!python experiments/sda-bench.py --help
!python experiments/sda-logreg.py --help




## `sda-bench.py` demo

`experiments/sda-bench.py` runs one registry objective through two solvers and saves one combined JSON summary.

Typical output locations:

- `outputs/sda-bench/<objective>/results.json`
- `outputs/sda-bench/<objective>/plots/png/*.png`
- `outputs/sda-bench/<objective>/plots/html/*.html`
- `outputs/sda-bench/<objective>/plots/manifest.json`


In [ ]:
!python experiments/sda-bench.py -o max_affine_4d -d 10 -g 1.0 --alpha 1.0 -i 100 -r



In [ ]:

!python outputs/generate_plots.py outputs/sda-bench/max_affine_4d/results.json




## Logistic-regression datasets

The logistic loader expects:

- numeric feature columns,
- binary labels in `{0, 1}`,
- a label column such as `y`, `target`, `label`, `labels`, or `class`.

Useful output locations for `experiments/sda-logreg.py`:

- `outputs/sda-logreg/<dataset-stem>/logreg/results.json`
- `outputs/sda-logreg/<dataset-stem>/logreg/plots/png/*.png`
- `outputs/sda-logreg/<dataset-stem>/logreg/plots/html/*.html`
- `outputs/sda-logreg/<dataset-stem>/logreg/plots/manifest.json`


In [ ]:
!python data/generate_logistic_data.py --help

synthetic_df = pd.read_csv(ROOT / 'data/synthetic_logistic.csv')
synthetic_df.head()


In [ ]:
iris_path = ROOT / 'data/iris.csv'
iris_df = pd.read_csv(iris_path)
iris_binary = iris_df[iris_df['target'].isin([0, 1])].copy()
iris_binary.to_csv(iris_path, index=False)

print(f'Rewrote {iris_path} with target classes 0 and 1 only.')
print('class counts:', iris_binary['target'].value_counts().sort_index().to_dict())
print('columns:', list(iris_binary.columns))

!python data/generate_logistic_data.py --n-samples 300 --dimension 3 --beta 1.5 -2.0 0.75 --intercept -0.2 --seed 7 --flip-prob 0.05 --output data/demo_logistic.csv

demo_df = pd.read_csv(ROOT / 'data/demo_logistic.csv')
display(Markdown('Generated data first, then used it as input for `sda-logreg.py`.'))
display(demo_df.head())

!python experiments/sda-logreg.py data/demo_logistic.csv -d 2 -g 1 --alpha 0.5 -i 80

display(Markdown('Default output path for this run: `outputs/sda-logreg/demo_logistic/logreg/results.json`.'))

!python experiments/sda-logreg.py synthetic_logistic.csv -d 2 -g 1 --alpha 0.5 -i 80
!python experiments/sda-logreg.py iris.csv -d 2 -g 1 --alpha 0.5 -i 80

demo_results = show_json_keys('outputs/sda-logreg/demo_logistic/logreg/results.json')
demo_summary = pd.DataFrame([
    {
        'method': run_summary['method'],
        'iterations': run_summary['iterations'],
        'train_loss': run_summary.get('final_train_loss_x_hat', run_summary.get('final_train_loss')),
        'test_accuracy': run_summary.get('final_test_accuracy_x_hat', run_summary.get('final_test_accuracy')),
        'nnz': run_summary.get('final_nonzero_count_x_hat', run_summary.get('final_nonzero_count')),
        'runtime_s': run_summary.get('total_runtime_seconds'),
    }
    for run_summary in demo_results
])
display(Markdown(
    'The logistic `results.json` contains `sda`, `subgradient`, and `sklearn` runs. '
    'Key fields include train/test loss, test accuracy, nonzero counts, parameter vectors, and runtime fields.'
))
display(demo_summary)

!python outputs/generate_plots.py outputs/sda-logreg/demo_logistic/logreg/results.json
!python outputs/generate_plots.py outputs/sda-logreg/synthetic_logistic/logreg/results.json
!python outputs/generate_plots.py outputs/sda-logreg/iris/logreg/results.json

logreg_manifest = show_json_keys('outputs/sda-logreg/demo_logistic/logreg/plots/manifest.json')
display(Markdown(
    'Companion HTML plots live in `outputs/sda-logreg/<dataset-stem>/logreg/plots/html/`, '
    'and `plots/manifest.json` describes the generated plot bundle.'
))

for png in [
    'outputs/sda-logreg/demo_logistic/logreg/plots/png/train_loss_iterations.png',
    'outputs/sda-logreg/demo_logistic/logreg/plots/png/test_accuracy_iterations.png',
    'outputs/sda-logreg/demo_logistic/logreg/plots/png/parameters_iterations.png',
]:
    show_png(png)
